In [ ]:
"""
Multivariate Weather Forecasting: ARIMA (VARMAX) + SARIMA (SARIMAX)
======================================================================
Predicts weather parameters (Max Temperature, Min Temperature,
Precipitation, Wind, Relative Humidity, Solar) from a Date + a
Latitude/Longitude pair.
 
MODELING APPROACH
------------------
- "Multivariate ARIMA": a single VARMAX model (Vector ARMA) per location,
  jointly modeling all weather parameters together as one vector time
  series. This is the true multivariate model -- it captures
  cross-correlations between e.g. temperature and humidity.
- "SARIMA": one SARIMAX model per weather parameter per location (SARIMAX
  itself models a single series; seasonality is handled per-variable).
 
Latitude/Longitude is NOT used as a regressor inside a given location's
model -- within one location's time series, lat/lon is constant, so it
carries no information for that model (a constant regressor is
degenerate/collinear with the intercept). Instead, lat/lon is used to
select WHICH location's trained model to use. If your data has multiple
distinct locations, a separate ARIMA and SARIMA model is fit per location.
 
Requires:
    pip install statsmodels pandas numpy scikit-learn
 
NOTE ON SEASONAL PERIOD:
This data is daily. A true annual seasonal cycle would need
seasonal_order period=365, but fitting SARIMAX with a 365-day seasonal
period on 30+ years of daily data is extremely slow (often
impractically slow) with statsmodels. SEASONAL_PERIOD defaults to 7
(weekly) below purely for a runnable demo -- for real annual-cycle
accuracy, either set SEASONAL_PERIOD=365 and be prepared for a long fit,
or resample the data to weekly/monthly frequency first.
"""

In [4]:
import warnings
warnings.filterwarnings("ignore")
 
import os
import glob
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
 
try:
    from statsmodels.tsa.statespace.varmax import VARMAX
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    STATSMODELS_AVAILABLE = True
except ImportError:
    STATSMODELS_AVAILABLE = False

In [17]:
DATA_FOLDER = "./WeatherDataFromAcrossAlaska/"  # folder containing one or more weather CSV files
DATE_COL = "Date"
LAT_COL = "Latitude"
LON_COL = "Longitude"
TARGET_COLS = [
    "Max Temperature", "Min Temperature", "Precipitation",
    "Wind", "Relative Humidity", "Solar",
]

In [6]:
ARIMA_ORDER = (2, 1, 2)              # (p, d, q) for the VARMAX model
SARIMA_ORDER = (1, 1, 1)             # (p, d, q) for each SARIMAX model
SEASONAL_PERIOD = 7                  # see note above -- 7 (weekly) for a fast demo
SARIMA_SEASONAL_ORDER = (1, 1, 1, SEASONAL_PERIOD)

In [19]:
# ------------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------------
def loadData(folder_path):
    """Reads and concatenates every CSV file in folder_path (e.g. one file
    per weather station/location, such as weatherdata-682-1478.csv), then
    parses Date and sorts by location then date.
    """
    csv_paths = sorted(glob.glob(os.path.join(folder_path, "*.csv")))
    if not csv_paths:
        raise FileNotFoundError(f"No CSV files found in {folder_path}")
 
    frames = []
    for path in csv_paths:
        frame = pd.read_csv(path)
        frame["__source_file__"] = os.path.basename(path)
        frames.append(frame)
        #print(f"Loaded {len(frame)} rows from {os.path.basename(path)}")
 
    df = pd.concat(frames, ignore_index=True)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    df = df.sort_values([LAT_COL, LON_COL, DATE_COL]).reset_index(drop=True)
    print(f"Total combined rows: {len(df)} from {len(csv_paths)} file(s)")
    return df

In [8]:
def _prepareLocationSeries(df, lat, lon, freq="D"):
    """Extracts one location's time series, reindexed to a continuous
    daily calendar (filling any gaps by time-based interpolation)."""
    subset = df[(df[LAT_COL] == lat) & (df[LON_COL] == lon)].copy()
    subset = subset.set_index(DATE_COL).sort_index()
    subset = subset.asfreq(freq)
    subset[TARGET_COLS] = subset[TARGET_COLS].interpolate(method="time").ffill().bfill()
    return subset

In [9]:
# ------------------------------------------------------------------
# 2. Split data (chronological -- NOT random, since this is time series)
# ------------------------------------------------------------------
def splitData(series_df, test_size=0.30):
    """Splits a single location's time-indexed series into the first 70%
    (train) and last 30% (test), preserving time order."""
    n = len(series_df)
    split_idx = int(n * (1 - test_size))
    train = series_df.iloc[:split_idx]
    test = series_df.iloc[split_idx:]
    return train, test

In [10]:
# ------------------------------------------------------------------
# 3. Create model (build, but do not fit) -- separate functions per type
# ------------------------------------------------------------------
def createARIMAModel(train_df, target_cols, order=ARIMA_ORDER):
    """Builds an unfitted multivariate ARIMA model (VARMAX), jointly
    modeling all target_cols as one vector time series."""
    if not STATSMODELS_AVAILABLE:
        print("NOTE: statsmodels is not installed (pip install statsmodels) "
              "-- skipping ARIMA model.")
        return None
 
    endog = train_df[target_cols]
    model = VARMAX(endog, order=order, trend="c")
    return model
 
 
def createSARIMAModel(train_df, target_cols, order=SARIMA_ORDER,
                       seasonal_order=SARIMA_SEASONAL_ORDER):
    """Builds a dict of unfitted SARIMAX models, one per target column
    (SARIMAX models a single series at a time)."""
    if not STATSMODELS_AVAILABLE:
        print("NOTE: statsmodels is not installed (pip install statsmodels) "
              "-- skipping SARIMA model.")
        return None
 
    models = {}
    for col in target_cols:
        models[col] = SARIMAX(
            train_df[col],
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
    return models

In [11]:
# ------------------------------------------------------------------
# 4. Train model (fit)
# ------------------------------------------------------------------
def trainModel(model, model_type):
    """Fits the model(s) created by createARIMAModel() / createSARIMAModel().
    Returns the fitted result object (ARIMA) or a dict of fitted results (SARIMA)."""
    if model is None:
        return None
 
    if model_type == "ARIMA":
        return model.fit(disp=False)
 
    elif model_type == "SARIMA":
        return {col: m.fit(disp=False) for col, m in model.items()}
 
    else:
        raise ValueError("model_type must be 'ARIMA' or 'SARIMA'")

In [12]:
# ------------------------------------------------------------------
# 5. Test model: forecast over the test period, store actual vs predicted
# ------------------------------------------------------------------
def testModel(fitted_model, test_df, target_cols, model_type):
    """Forecasts len(test_df) steps ahead and returns a dict of
    {variable_name: DataFrame(Actual, Predicted)}."""
    if fitted_model is None:
        return {}
 
    steps = len(test_df)
    results = {}
 
    if model_type == "ARIMA":
        forecast = fitted_model.forecast(steps=steps)
        forecast.index = test_df.index
        for col in target_cols:
            results[col] = pd.DataFrame({
                "Actual": test_df[col],
                "Predicted": forecast[col],
            })
 
    elif model_type == "SARIMA":
        for col in target_cols:
            pred = fitted_model[col].forecast(steps=steps)
            pred.index = test_df.index
            results[col] = pd.DataFrame({
                "Actual": test_df[col],
                "Predicted": pred,
            })
 
    else:
        raise ValueError("model_type must be 'ARIMA' or 'SARIMA'")
 
    return results

In [13]:
# ------------------------------------------------------------------
# 6. Evaluate + compare model accuracy
# ------------------------------------------------------------------
def _evaluateForecast(results_dict):
    """Computes MAE, RMSE, and MAPE per weather variable."""
    rows = []
    for col, df_ in results_dict.items():
        actual, predicted = df_["Actual"], df_["Predicted"]
        mae = mean_absolute_error(actual, predicted)
        rmse = np.sqrt(mean_squared_error(actual, predicted))
        nonzero = actual.replace(0, np.nan)
        mape = (np.abs((actual - predicted) / nonzero)).mean() * 100
        rows.append({"Variable": col, "MAE": mae, "RMSE": rmse, "MAPE (%)": mape})
    return pd.DataFrame(rows).set_index("Variable")

In [14]:
def compareModelAccuracy(arima_results, sarima_results):
    """Takes the per-variable results dicts from testModel() for both
    model types and returns a side-by-side comparison table."""
    arima_eval = _evaluateForecast(arima_results).add_prefix("ARIMA_") if arima_results else pd.DataFrame()
    sarima_eval = _evaluateForecast(sarima_results).add_prefix("SARIMA_") if sarima_results else pd.DataFrame()
 
    if arima_eval.empty and sarima_eval.empty:
        print("No model results to compare (statsmodels not installed?).")
        return pd.DataFrame()
 
    comparison = pd.concat([arima_eval, sarima_eval], axis=1)
    return comparison

In [15]:
# ------------------------------------------------------------------
# 7. Predict weather parameters for a new Date + Latitude/Longitude
# ------------------------------------------------------------------
def _findNearestLocation(df, lat, lon):
    """Finds the closest known (Latitude, Longitude) pair in the data to
    the requested one (exact match if available)."""
    locs = df[[LAT_COL, LON_COL]].drop_duplicates().copy()
    locs["dist"] = np.sqrt((locs[LAT_COL] - lat) ** 2 + (locs[LON_COL] - lon) ** 2)
    nearest = locs.sort_values("dist").iloc[0]
    return nearest[LAT_COL], nearest[LON_COL]

In [16]:
def predictNewData(date_input, lat_input, lon_input, df_full,
                    fitted_models_by_location, target_cols, model_type="ARIMA"):
    """Given a new date and Latitude/Longitude pair, forecasts the
    weather parameters and returns them as a single-row pandas DataFrame
    in the same column format as the original data (Date, Latitude,
    Longitude, <weather columns>).
    """
    nearest_lat, nearest_lon = _findNearestLocation(df_full, lat_input, lon_input)
    fitted_model = fitted_models_by_location.get((nearest_lat, nearest_lon))
    if fitted_model is None:
        raise ValueError(
            f"No trained {model_type} model available for location "
            f"({nearest_lat}, {nearest_lon})."
        )
 
    series = _prepareLocationSeries(df_full, nearest_lat, nearest_lon)
    last_date = series.index.max()
    target_date = pd.to_datetime(date_input)
    steps = (target_date - last_date).days
 
    if steps <= 0:
        # Requested date falls within (or before) the historical range
        if target_date in series.index:
            row = series.loc[[target_date], target_cols].copy()
        else:
            raise ValueError(
                f"{date_input} is within the historical range but not an "
                "exact observed date; only forecasting beyond the last "
                "known date is supported by this function."
            )
    else:
        if model_type == "ARIMA":
            forecast = fitted_model.forecast(steps=steps)
            row = forecast.iloc[[-1]][target_cols].copy()
            row.index = [target_date]
        elif model_type == "SARIMA":
            values = {}
            for col in target_cols:
                pred = fitted_model[col].forecast(steps=steps)
                values[col] = pred.iloc[-1]
            row = pd.DataFrame([values], index=[target_date])
        else:
            raise ValueError("model_type must be 'ARIMA' or 'SARIMA'")
 
    row.index.name = DATE_COL
    row = row.reset_index()
    row.insert(1, LAT_COL, nearest_lat)
    row.insert(2, LON_COL, nearest_lon)
    return row

In [21]:
def predictFromUserInput(df, arima_fitted_by_location, sarima_fitted_by_location):
    """Prompts the user for a date and Latitude/Longitude pair, then
    prints the predicted weather parameters as a DataFrame."""
    date_input = input("Enter a date (YYYY-MM-DD): ").strip()
    lat_input = float(input("Enter Latitude: ").strip())
    lon_input = float(input("Enter Longitude: ").strip())
    model_choice = input("Which model? (ARIMA/SARIMA) [ARIMA]: ").strip().upper() or "ARIMA"
 
    fitted_models_by_location = (
        arima_fitted_by_location if model_choice == "ARIMA" else sarima_fitted_by_location
    )
 
    prediction_df = predictNewData(
        date_input, lat_input, lon_input, df,
        fitted_models_by_location, TARGET_COLS, model_type=model_choice,
    )
    print("\nPredicted weather parameters:")
    print(prediction_df)
    return prediction_df

In [20]:
df = loadData(DATA_FOLDER)

Total combined rows: 454333 from 35 file(s)


In [22]:
arima_fitted_by_location = {}

In [23]:
sarima_fitted_by_location = {}

In [ ]:
for lat, lon in locations:
        print(f"\n=== Location ({lat}, {lon}) ===")
        series = _prepareLocationSeries(df, lat, lon)
        train, test = splitData(series, test_size=0.30)
        print(f"Train rows: {len(train)}  |  Test rows: {len(test)}")
 
        # --- ARIMA (VARMAX) ---
        arima_model = createARIMAModel(train, TARGET_COLS)
        arima_fitted = trainModel(arima_model, model_type="ARIMA")
        arima_results = testModel(arima_fitted, test, TARGET_COLS, model_type="ARIMA")
        arima_fitted_by_location[(lat, lon)] = arima_fitted
 
        # --- SARIMA (SARIMAX per variable) ---
        sarima_model = createSARIMAModel(train, TARGET_COLS)
        sarima_fitted = trainModel(sarima_model, model_type="SARIMA")
        sarima_results = testModel(sarima_fitted, test, TARGET_COLS, model_type="SARIMA")
        sarima_fitted_by_location[(lat, lon)] = sarima_fitted
 
        # --- Compare ---
        comparison = compareModelAccuracy(arima_results, sarima_results)
        print(comparison)
        comparison.to_csv(f"model_comparison_lat{lat}_lon{lon}.csv")
    return arima_fitted_by_location, sarima_fitted_by_location, df

In [ ]:
predictFromUserInput(full_df, arima_models, sarima_models)